# NB4: Annotation-Free Difficulty Oracle
**AbstractAttentionKernel vs MLP baseline on COCO val2017**

Pre-registered target: AUROC > 0.70 (MLP baseline: 0.560)

In [ ]:
!pip install openai-clip scikit-learn scipy -q

In [ ]:
import sys, os
from pathlib import Path

# Add working dir for nb4_lar_graph.py
for p in ['/kaggle/working'] + [str(x) for x in Path('/kaggle/input').glob('*') if x.is_dir()]:
    if Path(p, 'nb4_lar_graph.py').exists():
        sys.path.insert(0, p)
        print(f'Found nb4_lar_graph.py at: {p}')
        break

# Find COCO paths by directory name only — no file scanning
def find_coco_paths(base='/kaggle/input', max_depth=5):
    images_dir = None
    captions_json = None
    def _walk(path, depth):
        nonlocal images_dir, captions_json
        if depth > max_depth or (images_dir and captions_json):
            return
        try:
            entries = list(os.scandir(path))
        except PermissionError:
            return
        for e in entries:
            if e.is_dir(follow_symlinks=False):
                if e.name == 'val2017':
                    images_dir = e.path
                elif e.name == 'annotations':
                    cap = os.path.join(e.path, 'captions_val2017.json')
                    if os.path.isfile(cap):
                        captions_json = cap
                else:
                    _walk(e.path, depth + 1)
            if images_dir and captions_json:
                return
    _walk(base, 0)
    return images_dir, captions_json

IMAGES_DIR, CAPTIONS_JSON = find_coco_paths()

print(f'Images dir:    {IMAGES_DIR}')
print(f'Captions file: {CAPTIONS_JSON}')

if not IMAGES_DIR or not CAPTIONS_JSON:
    print('\nNot found. Top-level /kaggle/input contents:')
    for e in os.scandir('/kaggle/input'):
        print(' ', e.path)


In [ ]:
import nb4_lar_graph as nb4

pairs = nb4.load_coco_pairs(
    images_dir=IMAGES_DIR,
    captions_json=CAPTIONS_JSON,
    n=5000,
    seed=42
)
print(f'Loaded {len(pairs)} pairs')

In [ ]:
import torch, numpy as np
from nb4_lar_graph import build_nb4_graph, GraphState

def run_nb4_coco(pairs, seed=42, k=10):
    torch.manual_seed(seed)
    np.random.seed(seed)
    print(f'\n{"="*60}')
    print(f'  NB4  |  seed={seed}  k={k}  n={len(pairs)}')
    print(f'{"="*60}\n')
    entry, executor = build_nb4_graph(k=k)
    state = GraphState({'raw_pairs': pairs, 'n_pairs': len(pairs), 'seed': seed})
    node = entry
    while node is not None:
        node = node.execute(state)
    return dict(state)

result = run_nb4_coco(pairs, seed=42, k=10)

In [ ]:
import clip as _clip
from PIL import Image as PILImage
import torch, torch.nn.functional as F

# Fix 1: PIL image encoding
def _fixed_encode(self, x):
    self._load()
    if self._model is None:
        z = torch.randn(self.output_dim, device=nb4.DEVICE)
        return z / z.norm()
    with torch.no_grad():
        if isinstance(x, PILImage.Image):
            img_tensor = self._prep(x).unsqueeze(0).to(nb4.DEVICE)
            return self._model.encode_image(img_tensor).squeeze(0).float()
        elif hasattr(x, 'shape'):
            return self._model.encode_image(x.to(nb4.DEVICE)).float()
        else:
            return self._model.encode_text(
                _clip.tokenize([x]).to(nb4.DEVICE)
            ).squeeze(0).float()
nb4.CLIPEncoderNode.encode = _fixed_encode

# Fix 2: Semantic cosine D-score with calibrated tau
def _cosine_d_execute(self, state):
    z_imgs = state.get('z_imgs')
    z_txts = state.get('z_txts')
    z_i = F.normalize(z_imgs, dim=-1)
    z_t = F.normalize(z_txts, dim=-1)
    d_scores = 1 - (z_i * z_t).sum(dim=-1)
    self.tau = torch.quantile(d_scores, 0.70).item()
    labels = (d_scores >= self.tau).float()
    state.set('d_scores', d_scores)
    state.set('labels', labels)
    state.set('calibrated_tau', self.tau)
    n_hard = labels.sum().int().item()
    print(f'[D_ScoreLabeler] Cosine D-scores (V1-V6, τ={self.tau:.4f})...')
    print(f'  D-scores: min={d_scores.min():.3f} max={d_scores.max():.3f} | Hard cases: {n_hard}/{len(d_scores)} ({100*n_hard/len(d_scores):.1f}%)')
    return self.next_node
nb4.D_ScoreLabeler.execute = _cosine_d_execute

# Fix 3: Eval split uses outer seed (not hardcoded 42)
def _fixed_oracle_execute(self, state):
    hcl_K = state.get('hcl_K')
    hcl_V = state.get('hcl_V')
    z_imgs = state.get('z_imgs')
    d_scores = state.get('d_scores')
    tau = state.get('calibrated_tau', 0.3)
    n = len(z_imgs)
    seed = state.get('seed', 42)
    perm = torch.randperm(n, generator=torch.Generator().manual_seed(seed))
    eval_idx = perm[:n // 5]
    z_eval = z_imgs[eval_idx]
    d_eval = d_scores[eval_idx]
    y_true = (d_eval >= tau).cpu().numpy().astype(int)
    y_score_full = self._predict_full(z_eval, hcl_K, hcl_V)
    y_score_knn = self._predict_knn(z_eval, hcl_K, hcl_V)
    state.set('oracle_scores_full', y_score_full)
    state.set('oracle_scores_knn', y_score_knn)
    state.set('eval_labels', y_true)
    state.set('eval_idx', eval_idx)
    print(f'[HCLAttentionOracle] Running attention oracle (I1-I6, k={self.k})...')
    print(f'  Eval set: {len(eval_idx)} pairs | Hard: {y_true.sum()} | Easy: {(1-y_true).sum()}')
    return self.next_node
nb4.HCLAttentionOracle.execute = _fixed_oracle_execute

# Fix 4: Write paths
def _patched_checkpoint_init(self, checkpoint_path='/kaggle/working/nb4_checkpoint.json', next_node=None):
    self.path = checkpoint_path
    self.next_node = next_node
nb4.ResumeCheckpointNode.__init__ = _patched_checkpoint_init

def _patched_logger_init(self, secret=nb4.HMAC_SECRET, log_dir='/kaggle/working/nb4_lar_logs', next_node=None):
    self.secret = secret
    self.log_dir = log_dir
    self.next_node = next_node
nb4.ResultLoggerNode.__init__ = _patched_logger_init

In [ ]:
from scipy import stats

SEEDS = [42, 7, 13, 99, 2025]
oracle_aurocs, mlp_aurocs = [], []

for seed in SEEDS:
    r = run_nb4_coco(pairs, seed=seed, k=10)
    if r.get('auroc_oracle_knn') is not None:
        oracle_aurocs.append(r['auroc_oracle_knn'])
        mlp_aurocs.append(r.get('auroc_mlp', 0.560))

if oracle_aurocs:
    t, p = stats.ttest_rel(oracle_aurocs, mlp_aurocs)
    print(f'\n{"="*60}')
    print(f'  NB4 5-SEED FINAL RESULTS')
    print(f'{"="*60}')
    print(f'  Oracle AUROC:  {np.mean(oracle_aurocs):.4f} ± {np.std(oracle_aurocs):.4f}')
    print(f'  MLP AUROC:     {np.mean(mlp_aurocs):.4f} ± {np.std(mlp_aurocs):.4f}')
    print(f'  Δ AUROC:       {np.mean(oracle_aurocs)-np.mean(mlp_aurocs):+.4f}')
    print(f'  Paired t-test: t={t:.3f}  p={p:.4f}')
    verdict = 'CONFIRMED ✅' if np.mean(oracle_aurocs) > 0.70 else 'NOT CONFIRMED ❌'
    print(f'  Verdict:       {verdict}')
    print(f'{"="*60}')